## Setup

In [1]:
import sys
sys.path.insert(0, "../../src/python/")

In [2]:
import os
import pickle
import random
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

## Configuration

In [3]:
TRAIN_IMAGE_DIR = "/mnt/d/skin-lesion-data/final-data/train/"
TRAIN_LABEL_FILE = "/mnt/d/skin-lesion-data/final-data/labels/mappings.pkl"

VAL_IMAGE_DIR = "/mnt/d/skin-lesion-data/final-data/test/"
VAL_LABEL_FILE = "/mnt/d/skin-lesion-data/final-data/test_labels/mappings_test.pkl"

In [4]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUM_CLASSES = 8
IMAGE_SIZE = 224
BATCH_SIZE = 32

EPOCHS = 20

LR = 1e-4
WEIGHT_DECAY = 1e-4

## Dataset Class

In [5]:
CLASS_TO_IDX = {"MEL": 0, "NV": 1, "BCC": 2, "AK": 3, "BKL": 4, "DF": 5, "VASC": 6, "SCC": 7}

class SkinLesionDataset(Dataset):

    def __init__(self, image_dir, label_file, transform=None):

        self.image_dir = image_dir
        self.transform = transform

        with open(label_file, "rb") as f:
            self.labels = pickle.load(f)

        self.image_names = list(self.labels.keys())

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):

        image_name = self.image_names[idx]
        if "." not in image_name:
            image_name += ".jpg"
        image_path = os.path.join(self.image_dir, image_name)
        image = Image.open(image_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = self.labels[self.image_names[idx]]
        if isinstance(label, str):
            label = CLASS_TO_IDX[label]
        return image, label

## Transforms

In [6]:
train_transform = transforms.Compose([transforms.Resize((224,224)), transforms.RandomHorizontalFlip(), transforms.RandomRotation(15), transforms.ToTensor(),
                                      transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])])

val_transform = transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor(),transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])])

## Dataset and DataLoader

In [7]:
import random
from torch.utils.data import DataLoader

random.seed(42)
TARGET_COUNTS = {"MEL": 5000, "NV": 5000, "BCC": 3316, "AK": 1056, "BKL": 2829, "DF": 239, "VASC": 253, "SCC": 431}

train_dataset = SkinLesionDataset(TRAIN_IMAGE_DIR, TRAIN_LABEL_FILE, train_transform)
val_dataset = SkinLesionDataset(VAL_IMAGE_DIR, VAL_LABEL_FILE, val_transform)

class_to_images = {}

for img_name in train_dataset.image_names:
    label = train_dataset.labels[img_name]

    if label not in class_to_images:
        class_to_images[label] = []

    class_to_images[label].append(img_name)

balanced_images = []
for label, images in class_to_images.items():
    target = TARGET_COUNTS.get(label, len(images))
    if len(images) > target:
        balanced_images.extend(random.sample(images, target))
    else:
        balanced_images.extend(images)

random.shuffle(balanced_images)
train_dataset.image_names = balanced_images

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=16, pin_memory=True)

val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=16, pin_memory=True)

print("\nBalanced Training Distribution\n")

for cls in ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]:
    print(f"{cls:<5}: {TARGET_COUNTS[cls]}")

print()

print(f"Training Images   : {len(train_dataset)}")
print(f"Validation Images : {len(val_dataset)}")
print(f"Batch Size        : 32")
print(f"Workers           : 16")
print(f"Weighted Sampler  : False")
print(f"Downsampling      : True")


Balanced Training Distribution

MEL  : 5000
NV   : 5000
BCC  : 3316
AK   : 1056
BKL  : 2829
DF   : 239
VASC : 253
SCC  : 431

Training Images   : 18124
Validation Images : 8238
Batch Size        : 32
Workers           : 16
Weighted Sampler  : False
Downsampling      : True


/home/hrithik-dev/miniconda3/envs/skin-lesion-notebooks/lib/python3.10/site-packages/torch/utils/data/dataloader.py:557: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 8, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


## Model

In [8]:
weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights=weights)
model.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.classifier[1].in_features, NUM_CLASSES))

model = model.to(DEVICE)
print(model)

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

## Loss, Optimizer & Scheduler

In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

## Training

In [10]:
def train_one_epoch(model, loader):

    model.train()

    running_loss = 0
    preds = []
    labels = []

    for images, target in tqdm(loader):

        images = images.to(DEVICE)
        target = target.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, target)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        pred = outputs.argmax(1)
        preds.extend(pred.cpu().numpy())
        labels.extend(target.cpu().numpy())

    acc = accuracy_score(labels, preds)

    return running_loss/len(loader), acc

## Validation Function

In [11]:
def validate(model, loader):

    model.eval()

    running_loss = 0
    preds = []
    labels = []

    with torch.no_grad():

        for images, target in tqdm(loader):

            images = images.to(DEVICE)
            target = target.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, target)
            running_loss += loss.item()
            pred = outputs.argmax(1)
            preds.extend(pred.cpu().numpy())
            labels.extend(target.cpu().numpy())

    acc = accuracy_score(labels, preds)
    return (running_loss/len(loader), acc, labels, preds)

## Training Loop

In [12]:
best_acc = 0

for epoch in range(EPOCHS):

    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc, y_true, y_pred = validate(model, val_loader)
    scheduler.step()

    print()
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss : {train_loss:.4f}")
    print(f"Train Acc  : {train_acc:.4f}")
    print(f"Val Loss   : {val_loss:.4f}")
    print(f"Val Acc    : {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "efficientnet_b0_best.pth")

print()
print("Best Validation Accuracy :", best_acc)

  0%|                                                                                                                                                                                  | 0/258 [00:00<?, ?it/s]/home/hrithik-dev/miniconda3/envs/skin-lesion-notebooks/lib/python3.10/site-packages/torch/utils/data/dataloader.py:557: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 8, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 258/258 [03:20<00:00,  1.28it/s]



Epoch 1/20
Train Loss : 1.0777
Train Acc  : 0.6106
Val Loss   : 1.3300
Val Acc    : 0.5180


  0%|                                                                                                                                                                                  | 0/567 [00:00<?, ?it/s]/home/hrithik-dev/miniconda3/envs/skin-lesion-notebooks/lib/python3.10/site-packages/torch/utils/data/dataloader.py:557: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 8, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(
  2%|███▌                                                                                                                                                                     | 12/567 [00:53<40:55,  4.42s/it]


KeyboardInterrupt: 

## Load Best Model

In [ ]:
model.load_state_dict(torch.load("efficientnet_b0_best.pth", map_location=DEVICE))
model.eval()

## Evaluation

In [ ]:
_, _, y_true, y_pred = validate(model, val_loader)

print(classification_report(y_true, y_pred))

## Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm)
fig, ax = plt.subplots(figsize=(8,8))
disp.plot(ax=ax, xticks_rotation=45, cmap="Blues")
plt.show()

In [ ]:
import os

print(os.listdir(VAL_IMAGE_DIR)[:10])